In [1]:
!mamba install pandas numpy matplotlib seaborn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, seaborn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 8.177 seconds
  Name          Version  Build                Channel
-------------------------------------------------------------------
+ pandas        3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
+ patsy         1.0.3    py313h1804a44_0      emscripten-forge-4x
+ seaborn       0.13.2   hd8ed1ab_3           conda-forge
+ seaborn-base  0.13.2   pyhd8ed1ab_3         conda-forge
+ six           1.17.0   pyhe01879c_1         conda-forge
+ statsmodels   0.15.0   np23py313h07ed413_0  emscripten-forge-4x
- pip           26.2.2   pyh145f28c_0         conda-forge


In [2]:
#PART1:  DATA PREPARATION 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("customer_churn.csv")

print("FIRST 5 ROWS: ")
print(df.head())
print("ROWS, COLUMNS")
print(df.shape)
print("DATA INFORMATION")
print(df.info())
print("SUM OF MISSING VALUES:")
print(df.isnull().sum())

FIRST 5 ROWS: 
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies      

In [3]:
# Standardize column names.

df.columns = df.columns.str.lower().str.replace(' ', '_')

# Standardize string values.

categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

 # Fix data types and missing values (totalcharges).
tc = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

 # Encode the target variable (churn).
df.churn = (df.churn == 'yes').astype(int)


In [4]:
# Quick Verification Checklist
print(df.dtypes) 
print(df.isnull().sum())

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges        float64
churn                 int32
dtype: object
customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
mo

In [5]:
!mamba install scikit-learn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, seaborn, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 14.557 seconds
  Name                Version     Build                Channel
----------------------------------------------------------------------------
+ ca-certificates     2026.10.23  hbd8a1cb_0           conda-forge
+ certifi             2026.10.23  pyhd8ed1ab_3         conda-forge
+ charset-normalizer  3.5.1       pyhd8ed1ab_0         conda-forge
+ idna                3.20        pyh5ded981_0         conda-forge
+ joblib              1.6.0       py313h57e9779_0      emscripten-forge-4x
+ narwhals            2.29.0      pyh5ded981_0         conda-forge
+ requests            2.34.2      pyhcf101f3_0         conda-forge
+ scikit-learn        1.9.1       np23py313hde19dca_0  emscripten-forge-4x
+ setuptools          84.2.1      pyhe730653_0         conda-forge
+ threadpoolctl

In [6]:
# PART3 (Setting Up the Validation Framework)

# Split into full train (80%) and test (20%).
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

# Split full train into train (60%) and validation (20%)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

# Reset dataframe indices.
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Extract target arrays (y) and drop churn from features (X).
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

del df_train['churn']
del df_val['churn']
del df_test['churn']


In [7]:
# Quick Verification Checklist
print(len(df_train)) 
print(len(df_val)) 
print(len(df_test))

4225
1409
1409


In [8]:
# PART.4 (Exploratory Data Analysis)

 # Reset indices and check missing values
df_full_train = df_full_train.reset_index(drop=True)
df_full_train.isnull().sum()

# Inspect the target variable (churn) distribution.
df_full_train.churn.value_counts(normalize=True)
global_churn_rate = df_full_train.churn.mean()
round(global_churn_rate, 2)

# Separate numerical and categorical features.
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender', 'seniorcitizen', 'partner', 'dependents',
    'phoneservice', 'multiplelines', 'internetservice',
    'onlinesecurity', 'onlinebackup', 'deviceprotection',
    'techsupport', 'streamingtv', 'streamingmovies',
    'contract', 'paperlessbilling', 'paymentmethod'
]

# Check cardinality (unique values) in categorical columns
df_full_train[categorical].nunique()

print(df.churn.value_counts())
print("Global Churn Rate")
print(global_churn_rate)

churn
0    5174
1    1869
Name: count, dtype: int64
Global Churn Rate
0.26996805111821087


In [9]:
df_raw = pd.read_csv('customer_churn.csv') 
print(df_raw.columns.tolist())
print(df_raw['Churn'].unique())


['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
<StringArray>
['No', 'Yes']
Length: 2, dtype: str


In [10]:
df = pd.read_csv('customer_churn.csv')
print("Step 1 (Raw):", df['Churn'].unique())

df.columns = df.columns.str.lower().str.replace(' ', '_')
print("Step 2 (Lower col):", df['churn'].unique())

df['churn'] = df['churn'].str.strip().str.lower()
print("Step 3 (Cleaned str):", df['churn'].unique())

df['churn'] = (df['churn'] == 'yes').astype(int)
print("Step 4 (Encoded):", df['churn'].value_counts().to_dict())


Step 1 (Raw): <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Step 2 (Lower col): <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Step 3 (Cleaned str): <StringArray>
['no', 'yes']
Length: 2, dtype: str
Step 4 (Encoded): {0: 5174, 1: 1869}


In [11]:
#PART.5 Feature importance (risk ratios and mutual information)
# Step 1: Define Feature Lists & Global Churn

from IPython.display import display
from sklearn.metrics import mutual_info_score

# Define numerical and categorical columns
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender', 'seniorcitizen', 'partner', 'dependents',
    'phoneservice', 'multiplelines', 'internetservice',
    'onlinesecurity', 'onlinebackup', 'deviceprotection',
    'techsupport', 'streamingtv', 'streamingmovies',
    'contract', 'paperlessbilling', 'paymentmethod'
]

global_churn = df_full_train['churn'].mean()
print(f"Global Churn Rate: {global_churn:.4f}")

# Step 2: Risk Ratio & Difference (Categorical Features)

for col in categorical:
    print(f"FEATURE: {col.upper()}")
    
    df_group = df_full_train.groupby(col)['churn'].agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean'] / global_churn
    
    display(df_group)

# Step 3: Mutual Information (Categorical Features)

def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train['churn'])

mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi_sorted = mi.sort_values(ascending=False).to_frame(name='mutual_info_score')

print("\nMutual Information Ranking")
display(mi_sorted)

# Step 4: Correlation (Numerical Features)

corr = df_full_train[numerical].corrwith(df_full_train['churn']).to_frame(name='correlation')
corr['abs_correlation'] = corr['correlation'].abs()
corr_sorted = corr.sort_values(by='abs_correlation', ascending=False)

print("\nNumerical Feature Correlation with Churn")
display(corr_sorted[['correlation']])


Global Churn Rate: 0.2700
FEATURE: GENDER


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980


FEATURE: SENIORCITIZEN


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208


FEATURE: PARTNER


,mean,count,diff,risk
partner,,,,
no,0.329809,2932,0.059841,1.221659
yes,0.205033,2702,-0.064935,0.759472


... (26 outputs left out)


Mutual Information Ranking


,mutual_info_score
contract,0.098320
onlinesecurity,0.063085
techsupport,0.061032
internetservice,0.055868
onlinebackup,0.046923
deviceprotection,0.043453
paymentmethod,0.043210
streamingtv,0.031853
streamingmovies,0.031581
paperlessbilling,0.017589



Numerical Feature Correlation with Churn


,correlation
tenure,-0.351885
monthlycharges,0.196805
totalcharges,-0.196353


<!-- tenure (-0.3519): Strongest numerical signal. The longer a customer stays with the company, the less likely they are to leave. -->

In [17]:
#PART.6 One-Hot Encoding & Logistic Regression
# PART.7 Model Interpretation and Using the Model 

# 1. Train final model on df_full_train
dicts_full_train = df_full_train[categorical + numerical].to_dict(orient='records')
y_full_train = df_full_train['churn'].values

# Note: y_test already exists in memory, so we only convert df_test features
dicts_test = df_test[categorical + numerical].to_dict(orient='records')

dv_final = DictVectorizer(sparse=False)
X_full_train = dv_final.fit_transform(dicts_full_train)
X_test = dv_final.transform(dicts_test)

model_final = LogisticRegression(solver='liblinear', random_state=1)
model_final.fit(X_full_train, y_full_train)

# 2. Evaluate once on the unseen test set
y_pred_test = model_final.predict_proba(X_test)[:, 1]
test_accuracy = (y_test == (y_pred_test >= 0.5)).mean()
print(f"Final Test Accuracy: {test_accuracy:.4f}")

# 3. Predict on a single customer
customer_idx = 10
customer = dicts_test[customer_idx]
X_customer = dv_final.transform([customer])
churn_prob = model_final.predict_proba(X_customer)[0, 1]

print(f"\nCustomer Churn Probability: {churn_prob:.2%}")
print(f"Send Retention Email? {'YES' if churn_prob >= 0.5 else 'NO'}")
print(f"Actual Customer Status: {'Churned' if y_test[customer_idx] == 1 else 'Stayed'}")


Final Test Accuracy: 0.8126

Customer Churn Probability: 49.53%
Send Retention Email? NO
Actual Customer Status: Stayed


In [18]:
#PART.8 Evaluation Metrics
from sklearn.metrics import confusion_matrix, roc_auc_score, classification_report

# 1. Confusion Matrix
cm = confusion_matrix(y_val, churn_decision)
print("Confusion Matrix:\n", cm)

# 2. Classification Report (Precision, Recall, F1)
print("\nClassification Report:\n", classification_report(y_val, churn_decision))

# 3. ROC AUC Score (uses raw probabilities y_pred, not binary churn_decision)
auc = roc_auc_score(y_val, y_pred)
print(f"Validation ROC AUC: {auc:.4f}")


Confusion Matrix:
 [[923 100]
 [174 212]]

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.90      0.87      1023
           1       0.68      0.55      0.61       386

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

Validation ROC AUC: 0.8466
